# Short Term Memory 

In [ ]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langgraph.checkpoint.postgres import PostgresSaver  
from langchain_groq import ChatGroq

import os 
DB_URI = os.getenv("DATABASE_URL")

load_dotenv()

model = ChatGroq(model="openai/gpt-oss-120b")



In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()
agent = create_agent(
    model=model,
    tools=[],
    checkpointer=checkpointer,
)

thread_1 = {"configurable": {"thread_id": "user_101"}}

# Turn 1
res1 = agent.invoke(
    {"messages": [{"role": "user", "content": "Hi! My name is Asadullah AI."}]},
    config=thread_1,
)
print(res1["messages"][-1].content)

# Turn 2 — same thread_id remembers John
res2 = agent.invoke(
    {"messages": [{"role": "user", "content": "What is my name?"}]},
    config=thread_1,
)
print(res2["messages"][-1].content)

# Turn 3 — different thread_id has no memory of thread_1
thread_2 = {"configurable": {"thread_id": "user_202"}}
res3 = agent.invoke(
    {"messages": [{"role": "user", "content": "What is my name?"}]},
    config=thread_2,
)
print(res3["messages"][-1].content)



In [ ]:

with PostgresSaver.from_conn_string(DB_URI) as checkpointer:
    checkpointer.setup() # auto create tables in PostgreSQL

    agent = create_agent(
        model=model,
        checkpointer=checkpointer,
        system_prompt="You are a helpful assistant that can remember user information and retrieve it when asked."
    )

    response = agent.invoke(
        {"messages":[{"role":"user","content":"What is async/await in python"}]},
        config={"configurable":{"thread_id": "11"}},
    )

    print(response["messages"][-1].content)
    print(len(response["messages"]))